In [2]:
import pandas as pd

# Direct url to a clean, public 3-class Twitter sentiment dataset on GitHub
dataset_url = "https://raw.githubusercontent.com/satyamk28/Twitter-Sentiment-Analysis/master/twitter_training.csv"

# Load the CSV directly from the URL
# The dataset has 4 columns without a header row: ID, Entity, Sentiment, and Text
df = pd.read_csv(dataset_url, names=['id', 'entity', 'sentiment', 'text'])

# 1. Drop any rows where the text or sentiment label is missing
df = df.dropna(subset=['text', 'sentiment'])

# 2. Keep only the columns needed for training
df = df[['text', 'sentiment']]

# 3. Clean up the label formatting to lowercase
df['sentiment'] = df['sentiment'].str.lower()

# 4. Filter out any irrelevant tags if they exist, keeping just the 3 target classes
df = df[df['sentiment'].isin(['positive', 'negative', 'neutral'])]

# 5. Map the text sentiments to numeric integers for your LSTM model
label_mapping = {'negative': 0, 'neutral': 1, 'positive': 2}
df['label'] = df['sentiment'].map(label_mapping)

# Print out the shape and distribution to verify it loaded correctly
print(f"Dataset successfully loaded! Total rows: {df.shape[0]}")
print(df['sentiment'].value_counts())

Dataset successfully loaded! Total rows: 61121
sentiment
negative    22358
positive    20655
neutral     18108
Name: count, dtype: int64


In [3]:
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # Ensure text is treated as a string
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text


df_sample = df.sample(n=10000, random_state=42).reset_index(drop=True)

df_sample['cleaned_text'] = df_sample['text'].apply(clean_text)

# Tokenization and Padding
max_words = 5000
max_len = 30

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(df_sample['cleaned_text'])

sequences = tokenizer.texts_to_sequences(df_sample['cleaned_text'])
X = pad_sequences(sequences, maxlen=max_len, padding='post')
y = to_categorical(df_sample['label'], num_classes=3)

# Split into 80% Train, 20% Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Data preprocessed and split successfully!")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Data preprocessed and split successfully!


In [4]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, SpatialDropout1D
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

embedding_dim = 64

model = Sequential([
    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),
    SpatialDropout1D(0.2), # Dropout to prevent overfitting
    LSTM(units=64, dropout=0.2),
    Dense(units=3, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model
print("Training the LSTM model...")
model.fit(X_train, y_train, epochs=3, batch_size=32, validation_split=0.1, verbose=1)

# Evaluate
print("\nEvaluating the model performance...")
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

print(f"\nOverall Model Accuracy: {accuracy_score(y_true, y_pred):.4f}\n")
print(classification_report(y_true, y_pred, target_names=['Negative', 'Neutral', 'Positive']))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Training the LSTM model...
Epoch 1/3
225/225 ━━━━━━━━━━━━━━━━━━━━ 9s 22ms/step - accuracy: 0.4049 - loss: 1.0638 - val_accuracy: 0.5025 - val_loss: 0.9878
Epoch 2/3
225/225 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.5793 - loss: 0.8955 - val_accuracy: 0.6650 - val_loss: 0.7899
Epoch 3/3
225/225 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 0.7264 - loss: 0.6781 - val_accuracy: 0.6750 - val_loss: 0.7843

Evaluating the model performance...
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step

Overall Model Accuracy: 0.6615

              precision    recall  f1-score   support

    Negative       0.67      0.77      0.71       740
     Neutral       0.63      0.57      0.60       593
    Positive       0.68      0.62      0.65       667

    accuracy                           0.66      2000
   macro avg       0.66      0.65      0.65      2000
weighted avg       0.66      0.66      0.66      2000

